In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os, sys
ROOT = '/content/drive/MyDrive/DSP_Deconv'
for d in ['lib', 'data/raw', 'data/processed', 'results', 'figures', 'configs']:
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)
if f'{ROOT}/lib' not in sys.path:
    sys.path.insert(0, f'{ROOT}/lib')
print(os.listdir(ROOT))

['lib', 'data', 'results', 'figures', 'configs', 'test.ipynb', '01_forward_baselines.ipynb', '00_setup_data.ipynb']


In [ ]:
!pip -q install scikit-image tifffile pyyaml

import sys
import importlib

if "imp" not in sys.modules:
    sys.modules["imp"] = importlib

%load_ext autoreload
%autoreload 2

import numpy as np, scipy, skimage, matplotlib.pyplot as plt
print('numpy', np.__version__, '| scipy', scipy.__version__, '| skimage', skimage.__version__)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
numpy 2.1.3 | scipy 1.16.3 | skimage 0.25.2


In [ ]:
import yaml
CFG_PATH = f'{ROOT}/configs/params.yaml'
if not os.path.exists(CFG_PATH):
    cfg = dict(
        seeds=[0, 1, 2, 3, 4],
        psf_sigmas=[1.5, 2.5, 4.0],            
        gauss_noise=[0.005, 0.01, 0.02, 0.05],  
        poisson_peaks=[30, 100, 300],           
        crop=256,                               
        eps_band=1e-3,                          
    )
    yaml.safe_dump(cfg, open(CFG_PATH, 'w'))
cfg = yaml.safe_load(open(CFG_PATH))
cfg

{'crop': 256,
 'eps_band': 0.001,
 'gauss_noise': [0.005, 0.01, 0.02, 0.05],
 'poisson_peaks': [30, 100, 300],
 'psf_sigmas': [1.5, 2.5, 4.0],
 'seeds': [0, 1, 2, 3, 4]}

In [ ]:
import os, re, time, tarfile, gdown
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
if not os.path.exists('/content/fmd_repo'):
    !git clone -q https://github.com/yinhaoz/denoising-fluorescence.git /content/fmd_repo
script = open('/content/fmd_repo/denoising/download_dataset.sh').read()
ids = dict(re.findall(r'^(\w+)=["\']?([\w-]{20,})', script, flags=re.M))
wanted = ['WideField_BPAE_B', 'Confocal_BPAE_B', 'TwoPhoton_MICE']
out = f'{ROOT}/data/raw/FMD'

for name in wanted:
    tar_path = f'/content/{name}.tar'
    if not os.path.exists(tar_path):
        gdown.download(id=ids[name], output=tar_path, quiet=False)
    with tarfile.open(tar_path) as t:
        todo = [m for m in t.getmembers()
                if m.isfile() and ('/raw/' in m.name or '/gt/' in m.name)
                and not (os.path.exists(f'{out}/{m.name}')
                         and os.path.getsize(f'{out}/{m.name}') == m.size)]
        print(name, len(todo), 'files to write')
        for m in todo:
            for attempt in range(5):
                try:
                    os.makedirs(os.path.dirname(f'{out}/{m.name}'), exist_ok=True)
                    t.extract(m, out, filter='data')
                    break
                except FileNotFoundError:
                    time.sleep(2 * (attempt + 1))
            else:
                raise RuntimeError(f'gave up on {m.name}')
    print(name, 'done')

WideField_BPAE_B 1062 files to write
WideField_BPAE_B done
Confocal_BPAE_B 1062 files to write
Confocal_BPAE_B done
TwoPhoton_MICE 1062 files to write
TwoPhoton_MICE done


In [ ]:
from pathlib import Path
for p in sorted(Path(f'{ROOT}/data/raw/FMD').glob('*/*'))[:20]:
    print(p.relative_to(ROOT), len(list(p.iterdir())))

data/raw/FMD/TwoPhoton_MICE/raw 9


In [ ]:
from google.colab import files
files.upload()                                   # choose kaggle.json
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d paultimothymooney/kermany2018 -p /content/oct   # ~5 GB, to Colab's local disk, not Drive
!cd /content/oct && unzip -q kermany2018.zip "*test/NORMAL/*" "*test/DRUSEN/*"

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/paultimothymooney/kermany2018
License(s): CC-BY-NC-SA-4.0
kermany2018.zip: Skipping, found more recently modified local copy (use --force to force download)
replace OCT2017 /test/DRUSEN/DRUSEN-1083159-1.jpeg? [y]es, [n]o, [A]ll, [N]one, [r]ename: no
replace OCT2017 /test/DRUSEN/DRUSEN-1112835-1.jpeg? [y]es, [n]o, [A]ll, [N]one, [r]ename: no
replace OCT2017 /test/DRUSEN/DRUSEN-1173253-1.jpeg? [y]es, [n]o, [A]ll, [N]one, [r]ename: no
replace OCT2017 /test/DRUSEN/DRUSEN-1193659-1.jpeg? [y]es, [n]o, [A]ll, [N]one, [r]ename: none
replace OCT2017 /test/DRUSEN/DRUSEN-1219727-1.jpeg? [y]es, [n]o, [A]ll, [N]one, [r]ename: N


In [ ]:
import random, shutil
src = sorted(Path('/content/oct').rglob('test/*/*.jpeg'))
random.Random(0).shuffle(src)
dst = Path(f'{ROOT}/data/raw/OCT'); dst.mkdir(parents=True, exist_ok=True)
for p in src[:20]:
    shutil.copy(p, dst / p.name)
print(len(list(dst.iterdir())), 'OCT images saved')

20 OCT images saved
